# Ontologies to use
- **Post-Subreddit:** A post belongs to a specific subreddit.
- **Post-Author:** A post is created by an author.
- **Post-Topics:** A post is related to one or more topics based on the topic modeling results.
- **Post-Comments:** A post has a set of comments.
- **Post-Keywords:** A post is associated with specific keywords derived from the topic modeling.

## Loading libraries to be used

In [1]:
# Download NLTK resources 
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
!pip install pyvis



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import json
import rdflib
import gensim
import pandas as pd
import networkx as nx
from gensim import corpora
from pymongo import MongoClient
from nltk.corpus import stopwords
import plotly.graph_objects as go
from pyvis.network import Network
from collections import defaultdict
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

## Reading data from *MongoDB*

- Connect to MongoDB
- Read the data from the collection
- Convert the data into a pandas DataFrame
- Print the first few rows of the DataFrame

In [3]:
client = MongoClient("mongodb://localhost:27017/")
db = client['DataTails']
collection = db['Data']  
data_cursor = collection.find({})
DF = pd.DataFrame(list(data_cursor))
print(DF.head())

                        _id  type  \
0  66e965e698330736e0d693d5  None   
1  66e965e698330736e0d693d6  None   
2  66e965e698330736e0d693d7  None   
3  66e965e698330736e0d693d8  None   
4  66e965e698330736e0d693d9  None   

                                           postTitle postDesc  \
0  Adults(especially those over 30), how young do...      NaN   
1  What is a thing that your parents consider nor...      NaN   
2                 What is a smell that comforts you?      NaN   
3  When in history was it the best time to be a w...      NaN   
4    What's the worst way someone broke up with you?      NaN   

              postTime            authorName noOfUpvotes isNSFW  \
0  2024-08-06 01:02:35  Excellent-Studio7257        4068  False   
1  2024-08-06 01:47:22        Bigbumoffhappy        2073  False   
2  2024-08-05 22:21:53         bloomsmittenn        2188  False   
3  2024-08-06 03:32:59   More_food_please_77         778  False   
4  2024-08-05 21:01:39    ImpressiveWrap7363       

## Preprocessing the Data

- **Stopword Removal & Lemmatization:** 
    - Preprocessing() uses NLTK to remove stopwords and lemmatize the text.
- **Handling Missing Data:**
    - Missing postDesc fields are filled with an empty string.
    - Missing noOfUpvotes is filled with 0.
- **Datetime Conversion:** 
    - postTime is converted to a datetime object, and any errors are coerced.
- **Handling Comments:** 
    - Comments are converted from list format to a string of concatenated comments.
- **Final Clean Text:** 
    - Both postTitle and postDesc are cleaned using regular expressions and tokenization, and then passed through the NLTK-based text preprocessing function.

In [4]:
lemmatizer = WordNetLemmatizer()
StopWords = set(stopwords.words('english'))
def Preprocessing(text):
    text = re.sub(r'\W', ' ', text)
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in StopWords]
    return ' '.join(tokens)

Cols = ['subReddit', 'postTitle', 'postDesc', 'postTime', 'authorName', 'noOfUpvotes', 'comments', 'noOfComments', 'postUrl']
DF = DF[Cols]
print(DF.isnull().sum())
DF['postDesc'].fillna('', inplace=True)
DF['noOfUpvotes'].fillna(0, inplace=True)
DF['postTime'] = pd.to_datetime(DF['postTime'], errors='coerce')
DF['comments'] = DF['comments'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
DF['postTitle'] = DF['postTitle'].apply(lambda x: Preprocessing(str(x)))
DF['postDesc'] = DF['postDesc'].apply(lambda x: Preprocessing(str(x)))
print(DF.head())
print(DF.isnull().sum())

subReddit           2
postTitle           0
postDesc        15607
postTime            0
authorName        587
noOfUpvotes         0
comments            0
noOfComments        2
postUrl             2
dtype: int64
   subReddit                                          postTitle postDesc  \
0  AskReddit                  adult especially 30 young 18 seem            
1  AskReddit  thing parent consider normal consider normal a...            
2  AskReddit                                      smell comfort            
3  AskReddit                            history best time woman            
4  AskReddit                            worst way someone broke            

             postTime            authorName noOfUpvotes  \
0 2024-08-06 01:02:35  Excellent-Studio7257        4068   
1 2024-08-06 01:47:22        Bigbumoffhappy        2073   
2 2024-08-05 22:21:53         bloomsmittenn        2188   
3 2024-08-06 03:32:59   More_food_please_77         778   
4 2024-08-05 21:01:39    ImpressiveWr

## Topic Modeling

### Applying Topic Modeling on *Post Titles* and *Descriptions*
- **Subreddits as Main Topics:** 
    - We first group the posts based on their subreddit, treating each subreddit as a high-level topic or domain.
- **LDA for Subtopics:** 
    - For each post within a subreddit, we apply LDA to the combined postTitle and postDesc to discover finer subtopics.
- **Dictionary and Corpus:** 
    - For each subreddit, we create an LDA model, which generates a list of subtopics based on the tokenized posts.


In [5]:
GroupedData = DF.groupby('subReddit')
SubReddits = defaultdict(dict)
for subreddit, group in GroupedData:
    print(f"Processing Subreddit: {subreddit}")
    group['Combined'] = group['postTitle'] + " " + group['postDesc']
    group['Tokens'] = group['Combined'].apply(lambda x: word_tokenize(x))
    dictionary = corpora.Dictionary(group['Tokens'])
    corpus = [dictionary.doc2bow(text) for text in group['Tokens']]
    LDA = gensim.models.ldamodel.LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)
    subtopic_uris = [f"http://reddit.com/subtopic/{subreddit}/topic{topic[0]}" for topic in LDA.show_topics(formatted=False)]
    SubReddits[subreddit]['lda_model'] = LDA
    SubReddits[subreddit]['dictionary'] = dictionary
    SubReddits[subreddit]['corpus'] = corpus
    group['topics'] = [LDA.get_document_topics(bow) for bow in corpus]
    DF.loc[group.index, 'topics'] = group['topics']

# Now, DF will contain main topics (from subReddit) and subtopics (from LDA on postTitle + postDesc)


Processing Subreddit: Advice
Processing Subreddit: AmItheAsshole
Processing Subreddit: AskReddit
Processing Subreddit: Damnthatsinteresting
Processing Subreddit: Filmmakers
Processing Subreddit: Jokes
Processing Subreddit: Music
Processing Subreddit: NoStupidQuestions
Processing Subreddit: Showerthoughts
Processing Subreddit: askscience
Processing Subreddit: aww
Processing Subreddit: books
Processing Subreddit: funny
Processing Subreddit: gadgets
Processing Subreddit: gaming
Processing Subreddit: help
Processing Subreddit: islamabad
Processing Subreddit: memes
Processing Subreddit: mildlyinteresting
Processing Subreddit: movies
Processing Subreddit: news
Processing Subreddit: olympics
Processing Subreddit: pakistan
Processing Subreddit: pics
Processing Subreddit: politics
Processing Subreddit: programming
Processing Subreddit: science
Processing Subreddit: showerthoughts
Processing Subreddit: socialmedia
Processing Subreddit: sports
Processing Subreddit: technology
Processing Subreddit

## Building KG

- **Subreddit Nodes:** 
    - Each subreddit represents a main topic, and is connected to its posts via the belongs_to relationship.
- **Subtopic Nodes:** 
    - Each post is related to one or more subtopics (derived from LDA) via the related_to_subtopic relationship.
- **Other Nodes:** 
    - Authors, comments, and keywords are connected as before.

Reddit ontogolies
50 nodes
trending i.e chatgpt midjourney
longtail
Report
Algo of hot top


In [12]:
g = rdflib.Graph()

# Define namespaces
SIOC = rdflib.Namespace("http://rdfs.org/sioc/ns#")
DCMI = rdflib.Namespace("http://purl.org/dc/elements/1.1/")
FOAF = rdflib.Namespace("http://xmlns.com/foaf/0.1/")
g.bind("sioc", SIOC)
g.bind("dc", DCMI)
g.bind("foaf", FOAF)

# Function to add nodes to the graph
def add_post_to_graph(row, index, subtopic_uris):
    # Create URIs for each entity
    post_uri = rdflib.URIRef(f"http://reddit.com/post{index}")
    subreddit_uri = rdflib.URIRef(f"http://reddit.com/subreddit/{row['subReddit']}")
    author_uri = rdflib.URIRef(f"http://reddit.com/user/{row['authorName']}")

    # Add post details to the graph
    g.add((post_uri, rdflib.RDF.type, SIOC.Post))
    g.add((post_uri, DCMI.title, rdflib.Literal(row['postTitle'])))
    g.add((post_uri, DCMI.description, rdflib.Literal(row['postDesc'])))
    g.add((post_uri, DCMI.date, rdflib.Literal(row['postTime'])))
    g.add((post_uri, SIOC.num_replies, rdflib.Literal(row['noOfUpvotes'])))
    g.add((post_uri, SIOC.link, rdflib.URIRef(row['postUrl'])))

    # Link to subreddit
    g.add((subreddit_uri, rdflib.RDF.type, SIOC.Container))
    g.add((post_uri, SIOC.Container, subreddit_uri))
    g.add((subreddit_uri, SIOC.has_post, post_uri))

    # Link to author
    g.add((author_uri, rdflib.RDF.type, FOAF.Person))
    g.add((author_uri, FOAF.name, rdflib.Literal(row['authorName'])))
    g.add((post_uri, SIOC.has_creator, author_uri))

    # Add comments as nodes
    comment_uri = rdflib.URIRef(f"http://reddit.com/comment/{index}")
    g.add((comment_uri, rdflib.RDF.type, SIOC.Comment))
    g.add((comment_uri, DCMI.title, rdflib.Literal(row['comments'])))
    g.add((post_uri, SIOC.has_reply, comment_uri))

    # Link to subtopics (derived from LDA)
    for subtopic_uri in subtopic_uris:
        g.add((post_uri, rdflib.URIRef(subtopic_uri), rdflib.Literal("related_to_subtopic")))

# Select 20 nodes from each subreddit
selected_subreddits = ['Music', 'movies', 'pakistan', 'islamabad']

for subreddit in selected_subreddits:
    # Filter for the current subreddit
    subreddit_df = DF[DF['subReddit'] == subreddit].head(20)  # Select the first 20 posts for the subreddit

    # Add posts to graph
    for index, row in subreddit_df.iterrows():
        add_post_to_graph(row, index, subtopic_uris)

# Save graph in Turtle format or serialize as JSON-LD for D3.js
graph_file = 'D:/FYP/Github/data-tails/Backend/Ontologies/KG.ttl'
g.serialize(graph_file, format='turtle')

# Convert RDF graph to JSON-LD for D3 visualization
json_ld = g.serialize(format='json-ld', indent=4)
with open("D:/FYP/Github/data-tails/Backend/Ontologies/KG.json", "w", encoding="utf-8") as f:
    f.write(json_ld)

# D:/FYP/Github/data-tails/Backend/

### Converting *KG.json, KG.ttl* to format of D3 input file to view graph

In [13]:
import json
from collections import defaultdict

def transform_kg_to_d3(kg_file, d3_file):
    with open(kg_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Create sets to avoid duplicates and mappings for labels
    nodes_set = set()
    links_count = defaultdict(int)
    user_mapping = {}
    post_mapping = {}
    comment_mapping = {}

    # Add related_to_subtopic as a node (define it once)
    nodes_set.add("related_to_subtopic")  # Ensure subtopic node is added

    # Process each item in the KG.json
    for item in data:
        # Extract node ID from the "@id"
        node_id = item.get("@id")
        if node_id:
            nodes_set.add(node_id)

            # Label creation based on node type
            if "post" in node_id:
                post_label = node_id.split("/")[-1]  # Extracting post name
                post_mapping[node_id] = post_label
            elif "user" in node_id:
                user_label = node_id.split("/")[-1]  # Extracting username
                user_mapping[node_id] = user_label
            elif "comment" in node_id:
                comment_label = node_id.split("/")[-1]  # Extracting comment id
                comment_mapping[node_id] = comment_label

        # Create links based on relationships from different ontologies
        creator = item.get("http://rdfs.org/sioc/ns#has_creator")
        reply = item.get("http://rdfs.org/sioc/ns#has_reply")
        container = item.get("http://rdfs.org/sioc/ns#Container")
        subtopic_related = [item.get(f"ns1:topic{i}") for i in range(5)]  # Updated to handle ns1 topics
        
        # Handle SIOC relationships
        if creator and isinstance(creator, list) and creator:
            links_count[(node_id, creator[0]["@id"])] += 1
        if reply and isinstance(reply, list) and reply:
            links_count[(node_id, reply[0]["@id"])] += 1
        if container and isinstance(container, list) and container:
            links_count[(node_id, container[0]["@id"])] += 1

        # Handle subtopic relationships
        for subtopic in subtopic_related:
            if subtopic and isinstance(subtopic, list) and subtopic:
                links_count[(node_id, "related_to_subtopic")] += 1

    # Convert nodes set to a list of dictionaries with labels
    nodes_list = []
    for node in nodes_set:
        label = None
        if node in post_mapping:
            label = post_mapping[node]
        elif node in user_mapping:
            label = user_mapping[node]
        elif node in comment_mapping:
            label = comment_mapping[node]
        elif node == "related_to_subtopic":
            label = "Subtopic"  # Define label for related_to_subtopic

        # Append node with id and label if label exists
        nodes_list.append({
            "id": node,
            "label": label if label else node  # Fallback to id if label not found
        })

    # Create the final graph structure with links including their counts and types
    links_list = []
    for (source, target), count in links_count.items():
        relationship_type = None
        
        # Determine the relationship type based on the source and target node types
        if "user" in source and "post" in target:
            relationship_type = "created"
        elif "post" in source and "user" in target:
            relationship_type = "created_by"
        elif "user" in source and "comment" in target:
            relationship_type = "commented_on"
        elif "post" in source and "comment" in target:
            relationship_type = "has_comment"
        elif "comment" in source and "comment" in target:
            relationship_type = "reply_to"
        elif "user" in source and "user" in target:
            relationship_type = "knows"
        elif target == "related_to_subtopic":
            relationship_type = "related_to_subtopic"

        links_list.append({
            "source": source,
            "target": target,
            "value": count,
            "type": relationship_type
        })

    # Create the final graph structure
    graph = {
        "nodes": nodes_list,
        "links": links_list
    }

    # Write the transformed data to a new JSON file
    with open(d3_file, 'w', encoding='utf-8') as f:
        json.dump(graph, f, indent=2)

    print(f"Transformed data saved to {d3_file}")

# File paths
kg_file_path = "./Ontologies/KG.json"
d3_file_path = "./Ontologies/D3KG.json"

# Execute the transformation
transform_kg_to_d3(kg_file_path, d3_file_path)


Transformed data saved to ./Ontologies/D3KG.json
